# 01 · From snapshots to a benchmark

**Goal:** inspect the seven bundled datasets and the split used by each paper experiment. No training is required.

Install `requirements-dev.txt` first. The data are already included. The released biological H5AD files contain the **exact existing embeddings and time labels**, not expression matrices; nothing is recomputed here.

PACE receives unpaired point clouds. It does not receive cell correspondences or reference velocities.

In [1]:
from pathlib import Path
import os, sys
ROOT = Path.cwd().resolve()
if not (ROOT / 'train.py').exists():
    ROOT = ROOT.parent
assert (ROOT / 'train.py').exists(), 'Open this notebook from PACE/ or PACE/notebooks/.'
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
import torch
torch.set_num_threads(1)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
print('PACE workspace:', ROOT.name)


PACE workspace: PACE


In [2]:
from hydra import compose, initialize_config_dir
from src.methods.mfm.runner import build_datamodule
experiments = {
    'Ocean': 'oceans_positions_pace_ode',
    'Two-Branch': 'toy_bifurcation_pace_ode',
    'EB PHATE': 'eb_phate_pace_ode',
    'Schiebinger2019': 'wot_fle_three_interpolation_pace_ode',
    'iPSC-Liu': 'ipsc_liu_pace_ode',
    'OP-Cite': 'op_cite_pace_ode_100d',
    'OP-Multi': 'op_multi_pace_ode_100d',
}
configs, modules, rows = {}, {}, []
for title, experiment in experiments.items():
    with initialize_config_dir(config_dir=str(ROOT/'configs_hydra'), version_base='1.3'):
        cfg = compose(config_name='train', overrides=[f'experiment={experiment}', 'samples_per_timepoint=null'])
    dm = build_datamodule(cfg)
    dm.setup()
    configs[title], modules[title] = cfg, dm
    rows.append({'dataset': title, 'dimensions': cfg.dim,
                 'training labels': list(dm.unique_train_labels), 'held out': list(dm.unique_test_labels),
                 'selected train cells': sum(len(x) for x in dm.train_frames.values()),
                 'selected test cells': sum(len(x) for x in dm.test_frames.values())})
display(pd.DataFrame(rows))

,dataset,dimensions,training labels,held out,selected train cells,selected test cells
0,Ocean,2,"[0, 2, 4, 6, 8]","[1, 3, 5, 7]",555,444
1,Two-Branch,2,"[0, 2, 3, 5]","[1, 4]",256,128
2,EB PHATE,2,"[0, 1, 2, 4]",[3],13154,3665
3,Schiebinger2019,2,"[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 7.0, 8.0, 9.0, ...","[6.0, 11.0, 16.0]",73032,14082
4,iPSC-Liu,2,"[0, 8, 12, 20, 24]","[4, 16]",18292,10595
5,OP-Cite,100,"[2, 4, 7]",[3],24241,6999
6,OP-Multi,100,"[2, 4, 7]",[3],25157,8544


## Look at the time course

Filled points are training observations. Outlined points are held out. For 100D OP datasets we display **only the first two PCs** here; training still uses all 100 dimensions. Points are colored by time, never connected by assumed cell identities.

For this visualization we disabled configuration-level subsampling so the counts above reflect all cells at selected labels. The experiment presets retain their own training sample counts.

In [3]:
fig, axes = plt.subplots(2, 4, figsize=(15, 7))
rng = np.random.default_rng(42)
for ax, (title, dm) in zip(axes.flat, modules.items()):
    frames = {**dm.train_frames, **dm.test_frames}
    labels = sorted(frames)
    for i, label in enumerate(labels):
        points = frames[label].numpy()
        if len(points) > 800: points = points[rng.choice(len(points), 800, replace=False)]
        color = plt.cm.viridis(i/max(len(labels)-1, 1))
        if label in dm.unique_test_labels:
            ax.scatter(*points[:, :2].T, s=5, edgecolors=[color], facecolors='none', linewidths=.35)
        else:
            ax.scatter(*points[:, :2].T, s=3, color=color, alpha=.4)
    ax.set_title(title + (' · PC1/2 shown' if configs[title].dim > 2 else ''))
    ax.set_xticks([]); ax.set_yticks([])
axes.flat[-1].axis('off')
plt.tight_layout()
plt.show()

## Verify the release inputs

SHA-256 checksums detect a missing or modified file. The compact PCA exports also include a provenance sidecar with the original file hash and an exact-value verification statement.

In [4]:
import json, subprocess
subprocess.run([sys.executable, 'scripts/verify_data.py'], check=True)
provenance = json.loads((ROOT/'data/embeddings/ipsc_liu.provenance.json').read_text())
display({k: provenance[k] for k in ['cells', 'pca_dimensions', 'pca_dtype', 'validation']})

OK data/eb_velocity_v5.npz
OK data/embeddings/ipsc_liu.h5ad
OK data/embeddings/op_cite.h5ad
OK data/embeddings/op_multi.h5ad
OK data/oceans/oceans.npz
OK data/scnode/wot_fle/wot_fle.h5ad
OK data/toy_bifurcation/toy_bifurcation.csv
OK data/toy_bifurcation/toy_bifurcation.npz
8 inputs verified.


{'cells': 28887,
 'pca_dimensions': 100,
 'pca_dtype': 'float32',
 'validation': 'Exact array and obs equality after H5AD round-trip; no rows removed; no PCA refit.'}

### Next

Continue to [02 · Train and compare](02_train_and_compare.ipynb). For paper splits, including the separate 10D/50D iPSC holdouts, see [the experiment guide](../docs/experiments.md).